In [1]:

import random
from datetime import datetime, timedelta, timezone
from typing import Any

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from faker import Faker

In [20]:
def get_elastic_search(hosts: list, basic_auth: tuple[str, str]) -> Elasticsearch:
    return Elasticsearch(hosts=hosts, basic_auth=basic_auth, verify_certs=False)

In [21]:
elasticsearch = get_elastic_search(["http://elasticsearch:9200"], ("elastic", "elastic"))
print(elasticsearch.cluster.health()["status"], elasticsearch.cluster.health()["number_of_nodes"])

green 1


### Faker로 뉴스 데이터 생성

In [4]:
fake = Faker("ko_KR")
random.seed(42)

In [5]:
def make_article(i: int) -> dict:
    now = datetime.now(timezone.utc)
    published = now - timedelta(hours=random.randint(0, 240), minutes=random.randint(0, 59))
    subject = random.choice(["경제", "사회", "정치", "IT", "국제", "스포츠", "문화"])
    keyword = random.choice(["AI", "반도체", "환율", "증시", "부동산", "스타트업", "노동", "교육", "기후", "금리", "물가"])
    return {
        "id": i,
        "published": published.isoformat(),
        "subject": subject,
        "keyword": keyword,
        "title": fake.sentence(nb_words=10),
        "summary": " ".join(fake.paragraphs(nb=2)),
        "description": " ".join(fake.paragraphs(nb=6)),
        "original_link": fake.url(),
        "link": fake.url(),
        "created_at": now.isoformat(),
    }


def swap_alias(elastic: Elasticsearch, alias_: str, index_name_: str) -> dict[str, Any]:
    old_indices: list[str] = []
    if elastic.indices.exists_alias(name=alias_):
        old_indices = list(elastic.indices.get_alias(name=alias_).keys())

    actions_ = []
    for old in old_indices:
        if old != index_name_:
            actions_.append({"remove": {"index": old, "alias": alias_}})

    actions_.append({"add": {"index": index_name_, "alias": alias_}})
    elastic.indices.update_aliases(actions=actions_)
    removed = [x for x in old_indices if x != index_name_]
    return {"alias": alias_, "new_index": index_name_, "removed": removed, "actions": actions_}


def keep_last_n_indices_after_alias_swap(es: Elasticsearch, base_prefix: str, alias: str, keep: int = 3) -> dict:
    pattern = f"{base_prefix}-*"
    protected = set()
    if es.indices.exists_alias(name=alias):
        alias_map = es.indices.get_alias(name=alias)
        protected = set(alias_map.keys())

    settings = es.indices.get_settings(index=pattern, expand_wildcards="open")
    candidates = []
    for index_name, meta in settings.items():
        if index_name.startswith("."):
            continue
        if index_name in protected:
            pass
        creation_date = meta["settings"]["index"].get("creation_date")
        creation_date = int(creation_date) if creation_date is not None else 0
        candidates.append((index_name, creation_date))

    if not candidates:
        return {"kept": [], "deleted": [], "protected": list(protected)}

    candidates.sort(key=lambda x: x[1], reverse=True)
    keep_set = set([name for name, _ in candidates[:keep]])
    keep_set |= protected
    delete_list = [name for name, _ in candidates if name not in keep_set]

    deleted = []
    for idx in delete_list:
        if idx in protected:
            continue
        if es.indices.exists(index=idx):
            es.indices.delete(index=idx)
            deleted.append(idx)

    kept = [name for name, _ in candidates if name in keep_set]
    return {"kept": kept, "deleted": deleted, "protected": list(protected)}

## 인덱스 생성

In [6]:
base_index = "news_articles"
alias = "news_articles_current"
index_name = f"{base_index}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

index_body = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "id": {"type": "long"},
            "published": {"type": "date"},
            "subject": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}
            },
            "keyword": {"type": "keyword", "ignore_above": 256},
            "title": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}
            },
            "summary": {"type": "text"},
            "description": {"type": "text"},
            "original_link": {"type": "keyword", "ignore_above": 1024},
            "link": {"type": "keyword", "ignore_above": 1024},
            "created_at": {"type": "date"},
        }
    },
}


In [7]:
if elasticsearch.indices.exists(index=index_name):
    elasticsearch.indices.delete(index=index_name)

elasticsearch.indices.create(index=index_name, **index_body)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'news_articles-20260109140001'})

In [8]:
doc_count = 2000
actions = ({"_index": index_name, "_id": doc["id"], "_source": doc, } for doc in (make_article(i + 1) for i in range(doc_count)))
bulk(elasticsearch, actions, refresh="wait_for")

(2000, [])

### 조회 (간단 검색)

In [18]:
response_all = elasticsearch.search(index=index_name, size=5, query={"match_all": {}}, sort=[{"published": {"order": "desc"}}], )
print("total:", response_all["hits"]["total"]["value"])

for hit in response_all["hits"]["hits"]:
    src = hit["_source"]
    print(src["id"], src["published"], src["subject"], src["keyword"], src["title"], src["summary"][:10], src["description"][:100])

total: 2000
1086 2026-01-09T04:47:03.348763+00:00 IT 반도체 Aspernatur libero veritatis expedita aliquid nulla sed ad cumque. Voluptates Esse assumenda accusantium delectus commodi reiciendis. Ex ea animi. Ratione dolore repellendus aper
433 2026-01-09T04:43:01.324084+00:00 문화 환율 Dignissimos dolore harum vero suscipit molestiae. Eius facer Esse et corrupti enim ut reprehenderit. Totam labore reprehenderit. Est aliquam libero fugit consect
1499 2026-01-09T04:39:03.390175+00:00 정치 물가 Reprehenderit id error nihil voluptate totam praesentium doloremque rerum pariatur unde rerum. Omnis quib Ullam quidem nisi fuga. Quod dolores iure maiores. Asperiores ea facilis at amet voluptates. Tempore
1846 2026-01-09T04:27:04.386797+00:00 문화 환율 Sit itaque corporis rerum quaerat reiciendis fugiat repudiandae reiciendis dolor qui nisi. Laborum nu Amet consequatur nam perferendis vel animi. Esse rem deleniti deserunt. Distinctio excepturi dolorum
1169 2026-01-09T04:12:03.357012+00:00 스포츠 노동 Excepturi earum v

In [10]:
query_word = "Sunt ipsa debitis"
response = elasticsearch.search(
    index=index_name,
    size=5,
    query={
        "bool": {
            "must": [{"multi_match": {"query": query_word, "fields": ["title^2", "body"]}}],
            "filter": [{"terms": {"subject": ["IT", "경제", "문화"]}}],
        }
    },
    sort=[{"published": {"order": "desc"}}],
)

In [11]:
print(f"\n[search] index={index_name}, query='{query_word}' top5")

for hit in response["hits"]["hits"]:
    src = hit["_source"]
    print(f"- {src['published']} [{src['subject']}/{src['keyword']}] {src['title']} (id={src['id']})")


[search] index=news_articles-20260109140001, query='Sunt ipsa debitis' top5
- 2026-01-09T00:53:03.347819+00:00 [문화/환율] Asperiores accusantium eveniet debitis cumque doloremque veniam provident animi fugiat. (id=1077)
- 2026-01-08T18:46:04.369314+00:00 [문화/교육] Pariatur dolores excepturi minus sunt beatae sed labore eaque porro. (id=1652)
- 2026-01-08T17:16:04.364291+00:00 [문화/반도체] Modi odit dolorum pariatur dolor repellendus dicta quam sunt dolores dolorem. (id=1598)
- 2026-01-08T01:55:02.342380+00:00 [경제/부동산] Sunt nostrum perferendis molestias aspernatur nulla recusandae deleniti. (id=941)
- 2026-01-07T23:35:02.283609+00:00 [경제/스타트업] Vero eligendi mollitia libero blanditiis nihil dolores dolore ad reprehenderit eum cupiditate sunt. (id=506)


### Alias 연결 (스왑 방식)

In [12]:
swap_result = swap_alias(elasticsearch, alias, index_name)
print(swap_result)

{'alias': 'news_articles_current', 'new_index': 'news_articles-20260109140001', 'removed': ['news_articles-20260109135556'], 'actions': [{'remove': {'index': 'news_articles-20260109135556', 'alias': 'news_articles_current'}}, {'add': {'index': 'news_articles-20260109140001', 'alias': 'news_articles_current'}}]}


In [13]:
keep_result = keep_last_n_indices_after_alias_swap(elasticsearch, base_index, alias, keep=3)
print(keep_result)

{'kept': ['news_articles-20260109140001', 'news_articles-20260109135556', 'news_articles-20260109135456'], 'deleted': ['news_articles-20260109135424'], 'protected': ['news_articles-20260109140001']}


### Alias로 조회 (운영에서는 항상 alias로 조회)

In [14]:
response_alias = elasticsearch.search(index=alias, size=5, query={"match": {"summary": "perspiciatis"}}, sort=[{"published": {"order": "desc"}}])

for hit in response_alias["hits"]["hits"]:
    src = hit["_source"]
    print(f"- {src['published']} [{src['subject']}] {src['title']} (id={src['id']})")

- 2026-01-09T04:47:03.348763+00:00 [IT] Aspernatur libero veritatis expedita aliquid nulla sed ad cumque. (id=1086)
- 2026-01-09T03:40:04.367032+00:00 [사회] Labore rem commodi ullam veniam occaecati perferendis similique exercitationem accusamus rem voluptate fugit. (id=1628)
- 2026-01-09T03:36:04.374541+00:00 [스포츠] Adipisci expedita necessitatibus corporis sequi eveniet at. (id=1709)
- 2026-01-09T03:28:04.370627+00:00 [사회] Sed laudantium harum deleniti esse quasi amet natus saepe consectetur quam voluptas. (id=1666)
- 2026-01-09T02:53:04.361890+00:00 [스포츠] Error magnam dolorem consequatur molestiae aperiam vero praesentium possimus impedit recusandae nesciunt ea. (id=1573)
